# 批次轉換 k_time 為 datetime 格式

這個筆記本提供批次轉換所有 parquet 檔案中的 k_time 索引為 datetime 格式的功能。

In [9]:
import pandas as pd
import os
import glob
from datetime import datetime
import numpy as np
from IPython.display import display, HTML

## 步驟 1: 檢查現有檔案狀態

In [10]:
# 設定目錄路徑
data_dir = r'D:\03_預估量相關資量\tw_kbar_1m_vol_exp'

# 取得所有 parquet 檔案
pattern = os.path.join(data_dir, 'vol_exp_*.parquet')
files = glob.glob(pattern)
files = [f for f in files if '_backup' not in f]  # 排除備份檔案

print(f"找到 {len(files)} 個檔案")
print(f"\n前5個檔案：")
for f in files[:5]:
    print(f"  - {os.path.basename(f)}")

找到 108 個檔案

前5個檔案：
  - vol_exp_20250901.parquet
  - vol_exp_20250902.parquet
  - vol_exp_20250903.parquet
  - vol_exp_20250904.parquet
  - vol_exp_20250905.parquet


In [11]:
# 檢查第一個檔案的格式
if files:
    sample_file = files[0]
    df_sample = pd.read_parquet(sample_file)
    
    print(f"檔案: {os.path.basename(sample_file)}")
    print(f"\n索引資訊：")
    print(f"  - 名稱: {df_sample.index.name}")
    print(f"  - 類型: {df_sample.index.dtype}")
    print(f"  - 數量: {len(df_sample.index)}")
    
    print(f"\n前5個索引值：")
    for idx in df_sample.index[:5]:
        print(f"  - {idx}")
    
    print(f"\n資料形狀: {df_sample.shape}")
    print(f"股票數量: {len(df_sample.columns)}")

檔案: vol_exp_20250901.parquet

索引資訊：
  - 名稱: k_time
  - 類型: datetime64[us]
  - 數量: 266

前5個索引值：
  - 2025-09-01 09:00:00
  - 2025-09-01 09:01:00
  - 2025-09-01 09:02:00
  - 2025-09-01 09:03:00
  - 2025-09-01 09:04:00

資料形狀: (266, 531)
股票數量: 531


## 步驟 2: 定義轉換函數

In [12]:
def convert_time_to_datetime(time_str, date_str):
    """將時間字串轉換為 datetime 物件"""
    time_str = str(time_str).zfill(4)  # 確保是4位數
    
    # 解析日期
    year = int(date_str[:4])
    month = int(date_str[4:6])
    day = int(date_str[6:8])
    
    # 解析時間
    hour = int(time_str[:2])
    minute = int(time_str[2:4])
    
    return datetime(year, month, day, hour, minute)

# 測試轉換函數
test_date = '20250901'
test_times = ['0900', '0930', '1000', '1330']

print("轉換測試：")
for t in test_times:
    dt = convert_time_to_datetime(t, test_date)
    print(f"  {t} -> {dt}")

轉換測試：
  0900 -> 2025-09-01 09:00:00
  0930 -> 2025-09-01 09:30:00
  1000 -> 2025-09-01 10:00:00
  1330 -> 2025-09-01 13:30:00


## 步驟 3: 測試單一檔案轉換

In [13]:
def process_single_file(file_path, save=False):
    """處理單一檔案的轉換"""
    
    # 從檔案名稱提取日期
    filename = os.path.basename(file_path)
    date_str = filename.replace('vol_exp_', '').replace('.parquet', '')
    
    print(f"處理檔案: {filename}")
    print(f"日期: {date_str}")
    
    # 讀取資料
    df = pd.read_parquet(file_path)
    print(f"\n轉換前：")
    print(f"  索引名稱: {df.index.name}")
    print(f"  索引類型: {df.index.dtype}")
    print(f"  前3個索引: {df.index[:3].tolist()}")
    
    # 轉換索引
    if df.index.name == 'k_time':
        datetime_index = pd.DatetimeIndex([
            convert_time_to_datetime(t, date_str) for t in df.index
        ])
        df.index = datetime_index
        df.index.name = 'datetime'
        
        print(f"\n轉換後：")
        print(f"  索引名稱: {df.index.name}")
        print(f"  索引類型: {df.index.dtype}")
        print(f"  前3個索引: {list(df.index[:3])}")
        
        if save:
            # 備份原始檔案
            backup_path = file_path.replace('.parquet', '_backup.parquet')
            if not os.path.exists(backup_path):
                os.rename(file_path, backup_path)
                print(f"\n已備份至: {os.path.basename(backup_path)}")
            
            # 儲存新檔案
            df.to_parquet(file_path)
            print(f"已儲存轉換後的檔案")
        
        return df
    else:
        print(f"\n索引已經不是 k_time，跳過處理")
        return df

# 測試轉換（不儲存）
if files:
    df_converted = process_single_file(files[0], save=False)
    print(f"\n轉換後資料預覽：")
    display(df_converted.head())

處理檔案: vol_exp_20250901.parquet
日期: 20250901

轉換前：
  索引名稱: k_time
  索引類型: datetime64[us]
  前3個索引: [Timestamp('2025-09-01 09:00:00'), Timestamp('2025-09-01 09:01:00'), Timestamp('2025-09-01 09:02:00')]

轉換後：
  索引名稱: datetime
  索引類型: datetime64[ns]
  前3個索引: [Timestamp('2025-09-01 20:25:00'), Timestamp('2025-09-01 20:25:00'), Timestamp('2025-09-01 20:25:00')]

轉換後資料預覽：


,1101,1102,1210,1216,1229,1301,1303,1304,1305,1308,...,9103,9105,9904,9907,9914,9921,9933,9945,9955,9958
datetime,,,,,,,,,,,,,,,,,,,,,
2025-09-01 20:25:00,19231,1426,688,433,371,11661,33934,59,225,220,...,0,2785,2764,307,404,871,1239,1363,506,439
2025-09-01 20:25:00,15082,969,575,1470,356,9836,55356,105,175,160,...,0,3105,2077,1657,317,642,1481,1602,697,458
2025-09-01 20:25:00,14086,931,532,1725,772,9639,51304,81,176,131,...,0,3072,1844,1488,298,504,2688,1564,680,803
2025-09-01 20:25:00,12338,832,508,1596,986,8110,49792,78,144,124,...,0,2747,1809,1376,250,411,2786,1738,663,738
2025-09-01 20:25:00,12037,942,473,1406,1705,7234,48862,137,128,124,...,0,2598,1685,1288,215,359,2686,1696,741,885


## 步驟 4: 批次處理設定

In [14]:
# 處理選項
options = {
    'keep_backup': True,      # 是否保留備份
    'test_mode': True,        # 測試模式（只處理前3個檔案）
    'output_to_new_dir': False # 是否輸出到新目錄
}

# 如果要輸出到新目錄
if options['output_to_new_dir']:
    output_dir = r'D:\03_預估量相關資量\tw_kbar_1m_vol_exp_datetime'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"建立輸出目錄: {output_dir}")
else:
    output_dir = None

print("處理設定：")
for key, value in options.items():
    print(f"  - {key}: {value}")

處理設定：
  - keep_backup: True
  - test_mode: True
  - output_to_new_dir: False


## 步驟 5: 執行批次處理

In [15]:
# 注意：執行前請確認設定正確！
# 這個儲存格會實際修改檔案

def batch_process(files, test_mode=True, keep_backup=True, output_dir=None):
    """批次處理檔案"""
    
    if test_mode:
        files = files[:3]  # 測試模式只處理前3個
        print(f"測試模式：處理前 {len(files)} 個檔案\n")
    
    success_count = 0
    failed_count = 0
    results = []
    
    for i, file_path in enumerate(files, 1):
        print(f"\n[{i}/{len(files)}] {'-'*50}")
        
        try:
            filename = os.path.basename(file_path)
            date_str = filename.replace('vol_exp_', '').replace('.parquet', '')
            
            # 讀取並轉換
            df = pd.read_parquet(file_path)
            
            if df.index.name == 'k_time':
                # 轉換索引
                datetime_index = pd.DatetimeIndex([
                    convert_time_to_datetime(t, date_str) for t in df.index
                ])
                df.index = datetime_index
                df.index.name = 'datetime'
                
                # 決定輸出路徑
                if output_dir:
                    output_path = os.path.join(output_dir, filename)
                else:
                    output_path = file_path
                    # 備份原始檔案
                    if keep_backup:
                        backup_path = file_path.replace('.parquet', '_backup.parquet')
                        if not os.path.exists(backup_path):
                            os.rename(file_path, backup_path)
                
                # 儲存檔案
                df.to_parquet(output_path)
                
                success_count += 1
                results.append({'file': filename, 'status': '✓ 成功', 'note': '已轉換'})
                print(f"✓ {filename} - 轉換成功")
            else:
                results.append({'file': filename, 'status': '○ 跳過', 'note': '已經轉換過'})
                print(f"○ {filename} - 已經轉換過，跳過")
                
        except Exception as e:
            failed_count += 1
            results.append({'file': filename, 'status': '✗ 失敗', 'note': str(e)})
            print(f"✗ {filename} - 錯誤: {e}")
    
    # 顯示結果摘要
    print(f"\n{'='*60}")
    print(f"處理完成：")
    print(f"  成功: {success_count}")
    print(f"  失敗: {failed_count}")
    print(f"  總計: {len(files)}")
    
    # 建立結果表格
    df_results = pd.DataFrame(results)
    return df_results

# 執行批次處理（請確認設定後再執行）
# 取消註解下面這行來執行：
# results_df = batch_process(files, test_mode=True, keep_backup=True)
# display(results_df)

## 步驟 6: 驗證轉換結果

In [16]:
def verify_files(sample_count=3):
    """驗證檔案是否已成功轉換"""
    
    pattern = os.path.join(data_dir, 'vol_exp_*.parquet')
    files_to_check = glob.glob(pattern)
    files_to_check = [f for f in files_to_check if '_backup' not in f][:sample_count]
    
    results = []
    
    for file_path in files_to_check:
        filename = os.path.basename(file_path)
        try:
            df = pd.read_parquet(file_path)
            results.append({
                '檔案': filename,
                '索引名稱': df.index.name,
                '索引類型': str(df.index.dtype),
                '第一個時間': str(df.index[0]) if len(df.index) > 0 else 'N/A',
                '資料形狀': f"{df.shape[0]} x {df.shape[1]}"
            })
        except Exception as e:
            results.append({
                '檔案': filename,
                '索引名稱': 'Error',
                '索引類型': str(e),
                '第一個時間': 'N/A',
                '資料形狀': 'N/A'
            })
    
    df_verify = pd.DataFrame(results)
    return df_verify

# 驗證結果
print("驗證轉換結果：")
df_verification = verify_files(sample_count=5)
display(df_verification)

驗證轉換結果：


,檔案,索引名稱,索引類型,第一個時間,資料形狀
0,vol_exp_20250901.parquet,k_time,datetime64[us],2025-09-01 09:00:00,266 x 531
1,vol_exp_20250902.parquet,k_time,datetime64[us],2025-09-02 09:00:00,266 x 589
2,vol_exp_20250903.parquet,k_time,datetime64[us],2025-09-03 09:00:00,266 x 542
3,vol_exp_20250904.parquet,k_time,datetime64[us],2025-09-04 09:00:00,266 x 519
4,vol_exp_20250905.parquet,k_time,datetime64[us],2025-09-05 09:00:00,266 x 576
